<a href="https://colab.research.google.com/github/emanhassan2020/HandsOn/blob/main/Agents/HuggingFace/vision_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vision Agents with smolagents


This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

## Let's install the dependencies and login to our HF account to access the Inference API

If you haven't installed `smolagents` yet, you can do so by running the following command:

In [1]:
!pip install smolagents

Let's also login to the Hugging Face Hub to have access to the Inference API.

In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [7]:
!pip install --upgrade google-colab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 66.4 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.58.0
    Uninstalling google-auth-2.58.0:
      Successfully uninstalled google-auth-2.58.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 2.23.0 requires google-auth[requests]<3.0.0,>=2.56.0, but you have google-auth 2.49.0 which is incompatible.


In [4]:
!pip install  google-genai

In [7]:
!pip install google-generativeai

In [6]:
from google import genai
from google.colab import userdata

# Automatically retrieves the key from your Colab Secrets
api_key = userdata.get("GEMINI_API_KEY")

# Initialize the client
client = genai.Client(api_key=api_key)

# Generate a text response
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Explain Quantum Computing in one simple sentence."
)

print(response.text)

Quantum computing is a revolutionary technology that uses the bizarre rules of physics to explore millions of possibilities at once, solving complex problems far faster than any standard supercomputer ever could.


In [11]:
import os
import requests
from google.colab import userdata

# 1. Authenticate both APIs using Colab Secrets
try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    HF_TOKEN = userdata.get("HF_TOKEN")

    # Configure Gemini SDK
    genai.configure(api_key=GEMINI_API_KEY)
except Exception as e:
    print(f"Error loading secrets. Ensure keys match the exact names: {e}")

# 2. Use Gemini to generate a complex prompt or text layout
# Using the stable general purpose model
gemini_model = genai.GenerativeModel("gemini-3.5-flash")

prompt = "Write a highly descriptive, cinematic prompt for an AI image generator depicting a futuristic cyberpunk city at sunset."
print(f"--- Prompting Gemini: '{prompt}' ---\n")

gemini_response = gemini_model.generate_content(prompt)
generated_text = gemini_response.text
print(f"Gemini output text:\n{generated_text}\n")

--- Prompting Gemini: 'Write a highly descriptive, cinematic prompt for an AI image generator depicting a futuristic cyberpunk city at sunset.' ---

Gemini output text:
Here is a highly detailed, cinematic prompt optimized for AI image generators (like Midjourney v6, DALL-E 3, or Stable Diffusion XL):

### The Prompt

> **A breathtaking, cinematic wide-angle shot of a sprawling, multi-tiered cyberpunk megacity at sunset. Monolithic skyscrapers of dark steel and reflective glass stretch into a dramatic sky painted in gradients of fiery amber, dusty magenta, and deep violet. Sleek, aerodynamic flying vehicles leave long, luminous light trails as they weave between canyon-like towers. Giant, hyper-realistic 3D holographic advertisements—a glowing koi fish swimming through the air and a cybernetic model—cast a neon wash of hot pink, cyan, and electric blue over the urban landscape. A light, misty smog hangs in the air, catching volumetric god-rays from the dying sun. Far below, wet streets

## Providing Images at the Start of the Agent's Execution

In this approach, images are passed to the agent at the start and stored as `task_images` alongside the task prompt. The agent then processes these images throughout its execution.  

Consider the case where Alfred wants to verify the identities of the superheroes attending the party. He already has a dataset of images from previous parties with the names of the guests. Given a new visitor's image, the agent can compare it with the existing dataset and make a decision about letting them in.  

In this case, a guest is trying to enter, and Alfred suspects that this visitor might be The Joker impersonating Wonder Woman. Alfred needs to verify their identity to prevent anyone unwanted from entering.  

Let’s build the example. First, the images are loaded. In this case, we use images from Wikipedia to keep the example minimal, but imagine the possible use-cases!

In [15]:
from PIL import Image
import requests
from io import BytesIO

image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/e/e8/The_Joker_at_Wax_Museum_Plus.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/e/e8/The_Joker_at_Wax_Museum_Plus.jpg"
]
images = []
for url in image_urls:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
    }
    response = requests.get(url,headers=headers)
    image = Image.open(BytesIO(response.content)).convert("RGB")
    # image # This line is superfluous in this context as it just evaluates the variable without doing anything.
    images.append(image)

Now that we have the images, the agent will tell us wether the guests is actually a superhero (Wonder Woman) or a villian (The Joker).

In [12]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [13]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [14]:
from smolagents import CodeAgent, OpenAIModel, Tool

# Initialize Gemini via the OpenAI-compatible endpoint
model = OpenAIModel(
    model_id="gemini-3.5-flash",
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=GEMINI_API_KEY,
)
# Create and run your CodeAgent
agent = CodeAgent(model=model, tools=[])
agent.run("What is 20 * 5 plus 40?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is 20 * 5 plus 40?                                                                                         │
│                                                                                                                 │
╰─ OpenAIModel - gemini-3.5-flash ────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = 20 * 5 + 40                                                                                             
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 140

[Step 1: Duration 2.01 seconds| Input tokens: 2,161 | Output tokens: 21]

140

In [17]:
from smolagents import CodeAgent, OpenAIModel, Tool

# Initialize Gemini via the OpenAI-compatible endpoint
model = OpenAIModel(
    model_id="gemini-3.5-flash",
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=GEMINI_API_KEY,
)

#model = OpenAIServerModel(model_id="gpt-3.5-turbo")

# Instantiate the agent
agent = CodeAgent(
    tools=[],
    model=model,
    max_steps=20,
    verbosity_level=2
)

response = agent.run(
    """
    Describe the costume and makeup that the comic character in these photos is wearing and return the description.
    Tell me if the guest is The Joker or Wonder Woman.
    """,
    images=images
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Describe the costume and makeup that the comic character in these photos is wearing and return the description. │
│     Tell me if the guest is The Joker or Wonder Woman.                                                          │
│                                                                                                                 │
╰─ OpenAIModel - gemini-3.5-flash ────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
<code>description = """The comic character in the photos is wearing the following costume and makeup:              
                                                                                                                   
Makeup:                                                                                                            
- Face: Covered entirely in stark white clown-style greasepaint.                                                   
- Eyes: Highlighted with dark blue/grey eyeshadow.                                                                 
- Mouth: Features a highly exaggerated, wide, bright red smile painted over and far beyond the natural lips,       
curving upwards onto the cheeks.                                                                                   
- Hair: Slicked back with green dye/tint visible on the sides.                                                     
                                                                                                                   
Costume:                                                                                                           
- A textured purple jacket/coat with wide lapels.                                                                  
- An orange-gold (mustard-yellow) collared dress shirt.                                                            
- A shiny, purple satin-like cravat or wide necktie.                                                               
                                                                                                                   
Identity:                                                                                                          
The guest is The Joker."""                                                                                         
                                                                                                                   
final_answer(description)                                                                                          
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  description = """The comic character in the photos is wearing the following costume and makeup:                  
                                                                                                                   
  Makeup:                                                                                                          
  - Face: Covered entirely in stark white clown-style greasepaint.                                                 
  - Eyes: Highlighted with dark blue/grey eyeshadow.                                                               
  - Mouth: Features a highly exaggerated, wide, bright red smile painted over and far beyond the natural lips,     
  curving upwards onto the cheeks.                                                                                 
  - Hair: Slicked back with green dye/tint visible on the sides.                                                   
                                                                                                                   
  Costume:                                                                                                         
  - A textured purple jacket/coat with wide lapels.                                                                
  - An orange-gold (mustard-yellow) collared dress shirt.                                                          
  - A shiny, purple satin-like cravat or wide necktie.                                                             
                                                                                                                   
  Identity:                                                                                                        
  The guest is The Joker."""                                                                                       
                                                                                                                   
  final_answer(description)                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The comic character in the photos is wearing the following costume and makeup:

Makeup:
- Face: Covered entirely in stark white clown-style greasepaint.
- Eyes: Highlighted with dark blue/grey eyeshadow.
- Mouth: Features a highly exaggerated, wide, bright red smile painted over and far beyond the natural lips, 
curving upwards onto the cheeks.
- Hair: Slicked back with green dye/tint visible on the sides.

Costume:
- A textured purple jacket/coat with wide lapels.
- An orange-gold (mustard-yellow) collared dress shirt.
- A shiny, purple satin-like cravat or wide necktie.

Identity:
The guest is The Joker.

[Step 1: Duration 5.43 seconds| Input tokens: 4,388 | Output tokens: 164]

In [5]:
from smolagents import CodeAgent, OpenAIServerModel

model = OpenAIServerModel(model_id="gpt-3.5-turbo")

# Instantiate the agent
agent = CodeAgent(
    tools=[],
    model=model,
    max_steps=20,
    verbosity_level=2
)

response = agent.run(
    """
    Describe the costume and makeup that the comic character in these photos is wearing and return the description.
    Tell me if the guest is The Joker or Wonder Woman.
    """,
    images=images
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Describe the costume and makeup that the comic character in these photos is wearing and return the description. │
│     Tell me if the guest is The Joker or Wonder Woman.                                                          │
│                                                                                                                 │
╰─ OpenAIModel - gpt-3.5-turbo ───────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[Step 1: Duration 805.09 seconds]

KeyboardInterrupt: 

In [ ]:
response

{'description': '\n1. Costume:\n   - A purple suit with a yellow shirt and a large purple bow tie.\n   - Features a white flower lapel and a playing card in the second image.\n   - The style is flamboyant, consistent with a comic villain.\n\n2. Makeup:\n   - White face makeup covering the entire face.\n   - Red lips forming a wide, exaggerated smile.\n   - Blue eyeshadow with dark eye accents.\n   - Slicked-back green hair.\n',
 'character': 'The Joker'}

In this case, the output reveals that the person is impersonating someone else, so we can prevent The Joker from entering the party!

## Providing Images with Dynamic Retrieval

This examples is provided as a `.py` file since it needs to be run locally since it'll browse the web. Go to the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course) for more details.